In [8]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 0) SET PATH - GOOGLE COLAB
# ============================================================

BASE = Path("/content")
DATA = BASE
OUTPUT = BASE / "output"

OUTPUT.mkdir(exist_ok=True)

print("=" * 70)
print("TECHTROVE DATA INTEGRATION PIPELINE")
print("=" * 70)

print("\nCurrent folder:", BASE)
print("Output folder :", OUTPUT)


# ============================================================
# 1) CHECK FILES
# ============================================================

required_files = [
    "orders_2026_01.csv",
    "orders_2026_02.csv",
    "customers_crm.csv",
    "product_master.xlsx",
    "payments.json"
]

print("\n========== CHECK FILES ==========")

for filename in required_files:
    path = DATA / filename

    if not path.exists():
        raise FileNotFoundError(
            f"ไม่พบไฟล์: {path}"
        )

    print(filename, "-> OK")


# ============================================================
# 2) EXTRACT
# ============================================================

print("\n========== EXTRACT ==========")

jan = pd.read_csv(
    DATA / "orders_2026_01.csv"
)

feb = pd.read_csv(
    DATA / "orders_2026_02.csv"
)

customers = pd.read_csv(
    DATA / "customers_crm.csv"
)

products = pd.read_excel(
    DATA / "product_master.xlsx"
)

with open(
    DATA / "payments.json",
    "r",
    encoding="utf-8"
) as f:
    payments_raw = json.load(f)

payments = pd.json_normalize(
    payments_raw
)

payments = payments.rename(
    columns={
        "payment.method": "payment_method",
        "payment.status": "payment_status"
    }
)

print("January Orders :", jan.shape)
print("February Orders:", feb.shape)
print("Customers      :", customers.shape)
print("Products       :", products.shape)
print("Payments       :", payments.shape)


# ============================================================
# 3) RAW DATA PROFILING
# ============================================================

def profile_data(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nDtype:")
    print(df.dtypes)

    print("\nMissing:")
    display(
        df.isna().sum().to_frame("missing")
    )

    print(
        "\nDuplicate Rows:",
        df.duplicated().sum()
    )

    print("\nSample:")
    display(df.head())


profile_data(
    jan,
    "orders_2026_01.csv"
)

profile_data(
    feb,
    "orders_2026_02.csv"
)

profile_data(
    customers,
    "customers_crm.csv"
)

profile_data(
    products,
    "product_master.xlsx"
)

profile_data(
    payments,
    "payments.json"
)


# ============================================================
# 4) SAVE RAW QUALITY COUNTS
# ============================================================

raw_order_rows = len(jan) + len(feb)

raw_order_ids = pd.concat(
    [
        jan["order_id"],
        feb["order_id"]
    ],
    ignore_index=True
)

raw_duplicate_order_ids = (
    raw_order_ids.duplicated().sum()
)

raw_duplicate_customers = (
    customers["customer_id"]
    .duplicated()
    .sum()
)

raw_duplicate_payment_ids = (
    payments["payment_id"]
    .duplicated()
    .sum()
)

raw_missing_email = (
    customers["email"].isna().sum()
)

raw_missing_jan_price = (
    jan["unit_price"].isna().sum()
)

raw_missing_feb_price = (
    feb["unit_price"].isna().sum()
)

raw_payment_status_counts = (
    payments["payment_status"]
    .value_counts()
)

raw_orphan_payment_orders = (
    set(payments["order_id"])
    -
    set(raw_order_ids)
)


# ============================================================
# 5) SCHEMA ALIGNMENT
# ============================================================

print("\n========== SCHEMA ALIGNMENT ==========")

# ----------------------------
# January
# ----------------------------

jan = jan.rename(
    columns={
        "order_date": "order_datetime"
    }
)

# แปลงวันที่ January โดยใช้ ISO format โดยตรง
jan["order_datetime"] = pd.to_datetime(
    jan["order_datetime"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


# ----------------------------
# February
# ----------------------------

feb = feb.rename(
    columns={
        "ordered_at": "order_datetime",
        "qty": "quantity",
        "discount_pct": "discount"
    }
)

# แปลงวันที่ February เป็น DD/MM/YYYY
feb["order_datetime"] = pd.to_datetime(
    feb["order_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

# แปลง discount จาก 5% -> 0.05
feb["discount"] = (
    feb["discount"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
)

feb["discount"] = (
    pd.to_numeric(
        feb["discount"],
        errors="coerce"
    ) / 100
)

print("January columns:")
print(jan.columns.tolist())

print("\nFebruary columns:")
print(feb.columns.tolist())


# ============================================================
# 6) CONCAT
# ============================================================

print("\n========== CONCAT ==========")

orders = pd.concat(
    [
        jan,
        feb
    ],
    ignore_index=True
)

print(
    "Orders before concat:",
    raw_order_rows
)

print(
    "Orders after concat :",
    len(orders)
)


# ============================================================
# 7) CONVERT DATA TYPES
# ============================================================

orders["quantity"] = pd.to_numeric(
    orders["quantity"],
    errors="coerce"
)

orders["unit_price"] = pd.to_numeric(
    orders["unit_price"],
    errors="coerce"
)

orders["discount"] = pd.to_numeric(
    orders["discount"],
    errors="coerce"
)


# ============================================================
# 8) CLEAN ORDER TEXT
# ============================================================

for col in [
    "order_id",
    "customer_id",
    "product_id",
    "channel"
]:

    orders[col] = (
        orders[col]
        .astype("string")
        .str.strip()
    )


# ============================================================
# 9) DEDUPLICATION
# ============================================================

print("\n========== DEDUPLICATION ==========")

before_dedup = len(orders)

duplicate_order_rows = (
    orders["order_id"]
    .duplicated()
    .sum()
)

# เก็บข้อมูลล่าสุดตามลำดับที่ปรากฏ
orders = (
    orders
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .reset_index(drop=True)
)

after_dedup = len(orders)

print("Before       :", before_dedup)
print("Duplicate    :", duplicate_order_rows)
print("After        :", after_dedup)
print(
    "Rows removed :",
    before_dedup - after_dedup
)


# ============================================================
# 10) CLEAN CUSTOMER
# ============================================================

print("\n========== CLEAN CUSTOMER ==========")

customers["customer_id"] = (
    customers["customer_id"]
    .astype("string")
    .str.strip()
)

customers["full_name"] = (
    customers["full_name"]
    .astype("string")
    .str.strip()
)

customers["email"] = (
    customers["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

customers["province"] = (
    customers["province"]
    .astype("string")
    .str.strip()
)

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    errors="coerce"
)


# ============================================================
# 11) STANDARDIZE PROVINCE
# ============================================================

province_map = {

    "Bangkok":
        "กรุงเทพมหานคร",

    "กรุงเทพ":
        "กรุงเทพมหานคร",

    "กรุงเทพฯ":
        "กรุงเทพมหานคร",

    "กทม.":
        "กรุงเทพมหานคร",

    "Chonburi":
        "ชลบุรี",

    "ชลบุรี":
        "ชลบุรี",

    "ชลบุรี ":
        "ชลบุรี",

    "Chiang Mai":
        "เชียงใหม่",

    "เชียงใหม่":
        "เชียงใหม่",

    "เชียงใหม่ ":
        "เชียงใหม่",

    "Khon Kaen":
        "ขอนแก่น",

    "ขอนแก่น":
        "ขอนแก่น",

    "ขอนเเก่น":
        "ขอนแก่น",

    "ขอนแก่น ":
        "ขอนแก่น",

    "Rayong":
        "ระยอง",

    "ระยอง":
        "ระยอง",

    "ระยอง ":
        "ระยอง",

    "Phuket":
        "ภูเก็ต",

    "ภูเก็ต":
        "ภูเก็ต",

    "ภูเก็ต ":
        "ภูเก็ต"
}

customers["province"] = (
    customers["province"]
    .replace(province_map)
)


# ============================================================
# 12) CUSTOMER DUPLICATE
# ============================================================

customer_duplicate_rows = (
    customers["customer_id"]
    .duplicated()
    .sum()
)

customers = (
    customers
    .drop_duplicates(
        subset="customer_id",
        keep="last"
    )
    .reset_index(drop=True)
)

print(
    "Customer duplicate removed:",
    customer_duplicate_rows
)


# ============================================================
# 13) CLEAN PRODUCT
# ============================================================

print("\n========== CLEAN PRODUCT ==========")

products["product_id"] = (
    products["product_id"]
    .astype("string")
    .str.strip()
)

products["product_name"] = (
    products["product_name"]
    .astype("string")
    .str.strip()
)

products["category"] = (
    products["category"]
    .astype("string")
    .str.strip()
)

products["standard_price"] = pd.to_numeric(
    products["standard_price"],
    errors="coerce"
)

products["active_flag"] = (
    products["active_flag"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ============================================================
# 14) CLEAN PAYMENT
# ============================================================

print("\n========== CLEAN PAYMENT ==========")

payments["payment_id"] = (
    payments["payment_id"]
    .astype("string")
    .str.strip()
)

payments["order_id"] = (
    payments["order_id"]
    .astype("string")
    .str.strip()
)

payments["payment_method"] = (
    payments["payment_method"]
    .astype("string")
    .str.strip()
)

payments["payment_status"] = (
    payments["payment_status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

payments["paid_at"] = pd.to_datetime(
    payments["paid_at"],
    errors="coerce"
)


# duplicate payment_id
payments = (
    payments
    .drop_duplicates(
        subset="payment_id",
        keep="last"
    )
    .reset_index(drop=True)
)

# event ล่าสุดของแต่ละ order
payments = (
    payments
    .sort_values(
        ["order_id", "paid_at"]
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# 15) VALIDATE KEYS BEFORE MERGE
# ============================================================

customer_ids = set(
    customers["customer_id"].dropna()
)

product_ids = set(
    products["product_id"].dropna()
)

unmatched_customer_mask = (
    ~orders["customer_id"]
    .isin(customer_ids)
)

unmatched_product_mask = (
    ~orders["product_id"]
    .isin(product_ids)
)

unmatched_customer_rows = int(
    unmatched_customer_mask.sum()
)

unmatched_product_rows = int(
    unmatched_product_mask.sum()
)

unmatched_customer_ids = (
    orders.loc[
        unmatched_customer_mask,
        "customer_id"
    ]
    .dropna()
    .nunique()
)

unmatched_product_ids = (
    orders.loc[
        unmatched_product_mask,
        "product_id"
    ]
    .dropna()
    .nunique()
)

print("\n========== UNMATCHED KEYS ==========")

print(
    "Customer unmatched rows:",
    unmatched_customer_rows
)

print(
    "Customer unmatched IDs:",
    unmatched_customer_ids
)

print(
    "Product unmatched rows:",
    unmatched_product_rows
)

print(
    "Product unmatched IDs:",
    unmatched_product_ids
)


# ============================================================
# 16) MERGE CUSTOMER
# ============================================================

print("\n========== MERGE CUSTOMER ==========")

fact = orders.merge(
    customers[
        [
            "customer_id",
            "full_name",
            "email",
            "province",
            "signup_date"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_customer_match"
)

print(
    fact["_customer_match"]
    .value_counts()
)


# ============================================================
# 17) MERGE PRODUCT
# ============================================================

print("\n========== MERGE PRODUCT ==========")

fact = fact.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
            "standard_price",
            "active_flag"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="_product_match"
)

print(
    fact["_product_match"]
    .value_counts()
)


# ============================================================
# 18) MERGE PAYMENT
# ============================================================

print("\n========== MERGE PAYMENT ==========")

fact = fact.merge(
    payments[
        [
            "order_id",
            "payment_id",
            "payment_method",
            "payment_status",
            "paid_at"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_payment_match"
)

print(
    fact["_payment_match"]
    .value_counts()
)


# ============================================================
# 19) BUSINESS RULE VALIDATION
# ============================================================

print("\n========== BUSINESS RULES ==========")

valid_quantity = (
    fact["quantity"].notna()
    &
    (fact["quantity"] > 0)
)

valid_unit_price = (
    fact["unit_price"].notna()
    &
    (fact["unit_price"] > 0)
)

valid_discount = (
    fact["discount"].notna()
    &
    fact["discount"].between(
        0,
        1,
        inclusive="both"
    )
)

valid_customer = (
    fact["_customer_match"] == "both"
)

valid_product = (
    fact["_product_match"] == "both"
)

valid_payment = (
    fact["payment_status"] == "PAID"
)


print(
    "Quantity > 0:",
    int(valid_quantity.sum())
)

print(
    "Unit price > 0:",
    int(valid_unit_price.sum())
)

print(
    "Discount 0-1:",
    int(valid_discount.sum())
)

print(
    "Customer matched:",
    int(valid_customer.sum())
)

print(
    "Product matched:",
    int(valid_product.sum())
)

print(
    "Payment PAID:",
    int(valid_payment.sum())
)


# ============================================================
# 20) INVALID ROW COUNTS
# ============================================================

invalid_quantity_rows = int(
    (
        fact["quantity"].isna()
        |
        (fact["quantity"] <= 0)
    ).sum()
)

invalid_unit_price_rows = int(
    (
        fact["unit_price"].isna()
        |
        (fact["unit_price"] <= 0)
    ).sum()
)

invalid_discount_rows = int(
    (
        fact["discount"].isna()
        |
        ~fact["discount"].between(
            0,
            1,
            inclusive="both"
        )
    ).sum()
)


# ============================================================
# 21) NET SALES
# ============================================================

fact["net_sales"] = np.where(
    (
        valid_quantity
        &
        valid_unit_price
        &
        valid_discount
        &
        valid_customer
        &
        valid_product
        &
        valid_payment
    ),
    fact["quantity"]
    *
    fact["unit_price"]
    *
    (1 - fact["discount"]),
    0.0
)


# ============================================================
# 22) FACT SALES
# ============================================================

fact_sales = fact[
    (
        valid_quantity
        &
        valid_unit_price
        &
        valid_discount
        &
        valid_customer
        &
        valid_product
        &
        valid_payment
    )
].copy()

fact_sales["net_sales"] = (
    fact_sales["net_sales"]
    .round(2)
)

fact_sales = fact_sales[
    [
        "order_id",
        "order_datetime",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount",
        "channel",
        "payment_id",
        "payment_method",
        "payment_status",
        "paid_at",
        "province",
        "category",
        "standard_price",
        "active_flag",
        "net_sales"
    ]
].reset_index(drop=True)


# ============================================================
# 23) DIM CUSTOMER
# ============================================================

dim_customer = customers[
    [
        "customer_id",
        "full_name",
        "email",
        "province",
        "signup_date"
    ]
].copy()


# ============================================================
# 24) DIM PRODUCT
# ============================================================

dim_product = products[
    [
        "product_id",
        "product_name",
        "category",
        "standard_price",
        "active_flag"
    ]
].copy()


# ============================================================
# 25) SUMMARY BY PROVINCE
# ============================================================

summary_by_province = (
    fact_sales
    .groupby(
        "province",
        as_index=False
    )
    .agg(
        transactions=(
            "order_id",
            "nunique"
        ),
        total_quantity=(
            "quantity",
            "sum"
        ),
        net_sales=(
            "net_sales",
            "sum"
        )
    )
    .sort_values(
        "net_sales",
        ascending=False
    )
    .reset_index(drop=True)
)

summary_by_province["net_sales"] = (
    summary_by_province["net_sales"]
    .round(2)
)


# ============================================================
# 26) SUMMARY BY CATEGORY
# ============================================================

summary_by_category = (
    fact_sales
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        transactions=(
            "order_id",
            "nunique"
        ),
        total_quantity=(
            "quantity",
            "sum"
        ),
        net_sales=(
            "net_sales",
            "sum"
        )
    )
    .sort_values(
        "net_sales",
        ascending=False
    )
    .reset_index(drop=True)
)

summary_by_category["net_sales"] = (
    summary_by_category["net_sales"]
    .round(2)
)


# ============================================================
# 27) PAYMENT COUNTS
# ============================================================

failed_payment_rows = int(
    (
        fact["payment_status"] == "FAILED"
    ).sum()
)

refunded_payment_rows = int(
    (
        fact["payment_status"] == "REFUNDED"
    ).sum()
)

unmatched_payment_rows = int(
    (
        fact["_payment_match"] == "left_only"
    ).sum()
)


# ============================================================
# 28) DATA QUALITY REPORT
# ============================================================

data_quality_report = pd.DataFrame({

    "stage": [

        "raw",
        "raw",
        "raw",
        "raw",
        "raw",
        "raw",
        "raw",

        "cleaned",
        "cleaned",
        "cleaned",
        "cleaned",
        "cleaned",

        "validation",
        "validation",
        "validation",
        "validation",

        "final",
        "final",
        "final",
        "final",
        "final"
    ],

    "metric": [

        "raw_orders",
        "duplicate_order_ids",
        "duplicate_customer_ids",
        "duplicate_payment_ids",
        "missing_customer_emails",
        "missing_unit_price_january",
        "missing_unit_price_february",

        "duplicate_orders_removed",
        "duplicate_customer_rows_removed",
        "duplicate_payment_ids_removed",
        "invalid_quantity_rows",
        "invalid_unit_price_rows",

        "invalid_discount_rows",
        "unmatched_customer_rows",
        "unmatched_product_rows",
        "unmatched_payment_rows",

        "failed_payment_rows",
        "refunded_payment_rows",
        "valid_paid_sales",
        "total_net_sales",
        "orders_after_dedup"
    ],

    "value": [

        raw_order_rows,
        raw_duplicate_order_ids,
        raw_duplicate_customers,
        raw_duplicate_payment_ids,
        raw_missing_email,
        raw_missing_jan_price,
        raw_missing_feb_price,

        duplicate_order_rows,
        customer_duplicate_rows,
        raw_duplicate_payment_ids,
        invalid_quantity_rows,
        invalid_unit_price_rows,

        invalid_discount_rows,
        unmatched_customer_rows,
        unmatched_product_rows,
        unmatched_payment_rows,

        failed_payment_rows,
        refunded_payment_rows,
        len(fact_sales),
        round(
            fact_sales["net_sales"].sum(),
            2
        ),
        len(orders)
    ]
})


# ============================================================
# 29) SAVE OUTPUT
# ============================================================

dim_customer.to_csv(
    OUTPUT / "dim_customer.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_product.to_csv(
    OUTPUT / "dim_product.csv",
    index=False,
    encoding="utf-8-sig"
)

fact_sales.to_csv(
    OUTPUT / "fact_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

data_quality_report.to_csv(
    OUTPUT / "data_quality_report.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_by_province.to_csv(
    OUTPUT / "summary_by_province.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_by_category.to_csv(
    OUTPUT / "summary_by_category.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 30) FINAL RESULT
# ============================================================

total_net_sales = round(
    fact_sales["net_sales"].sum(),
    2
)

top_province = (
    summary_by_province.iloc[0]
)

top_category = (
    summary_by_category.iloc[0]
)

print("\n")
print("=" * 70)
print("                    FINAL OUTPUT")
print("=" * 70)

print(
    f"Raw Orders              : {raw_order_rows:,}"
)

print(
    f"After Dedup             : {len(orders):,}"
)

print(
    f"Duplicate Removed       : "
    f"{duplicate_order_rows:,}"
)

print(
    f"Customer Unmatched Rows : "
    f"{unmatched_customer_rows:,}"
)

print(
    f"Product Unmatched Rows  : "
    f"{unmatched_product_rows:,}"
)

print(
    f"Valid Paid Sales        : "
    f"{len(fact_sales):,}"
)

print(
    f"Total Net Sales         : "
    f"{total_net_sales:,.2f} บาท"
)

print(
    f"Top Province            : "
    f"{top_province['province']}"
)

print(
    f"Top Province Sales      : "
    f"{top_province['net_sales']:,.2f} บาท"
)

print(
    f"Top Category            : "
    f"{top_category['category']}"
)

print(
    f"Top Category Sales      : "
    f"{top_category['net_sales']:,.2f} บาท"
)


# ============================================================
# 31) SHOW TABLES
# ============================================================

print("\n========== SUMMARY BY PROVINCE ==========")
display(
    summary_by_province
)

print("\n========== SUMMARY BY CATEGORY ==========")
display(
    summary_by_category
)

print("\n========== DATA QUALITY REPORT ==========")
display(
    data_quality_report
)

print("\n========== FACT SALES (10 ROWS) ==========")
display(
    fact_sales.head(10)
)

print("\n========== DIM CUSTOMER (10 ROWS) ==========")
display(
    dim_customer.head(10)
)

print("\n========== DIM PRODUCT (10 ROWS) ==========")
display(
    dim_product.head(10)
)


# ============================================================
# 32) ANSWERS TO 6 QUESTIONS
# ============================================================

print("\n")
print("=" * 70)
print("                    6 ANALYSIS ANSWERS")
print("=" * 70)

print(
    f"""
7. หลังรวมไฟล์ orders มี {raw_order_rows:,} แถว
   และเหลือ {len(orders):,} แถวหลังลบ duplicate

8. มี customer_id ที่ไม่พบใน Master Data
   จำนวน {unmatched_customer_rows:,} แถว
   และมี product_id ที่ไม่พบ
   จำนวน {unmatched_product_rows:,} แถว

9. มียอดขายที่ใช้ได้จริง
   จำนวน {len(fact_sales):,} ธุรกรรม
   ยอดขายสุทธิรวม {total_net_sales:,.2f} บาท

10. จังหวัดที่มียอดขายสุทธิสูงสุดคือ
    {top_province['province']}
    ยอดขาย {top_province['net_sales']:,.2f} บาท

11. หมวดสินค้าที่มียอดขายสุทธิสูงสุดคือ
    {top_category['category']}
    ยอดขาย {top_category['net_sales']:,.2f} บาท

12. หากสลับลำดับ merge ก่อน cleaning
    ความน่าเชื่อถือของข้อมูลอาจลดลง เพราะข้อมูลดิบยังมี
    duplicate key, รูปแบบข้อความที่ไม่เป็นมาตรฐาน,
    schema ที่แตกต่างกันระหว่างเดือน และข้อมูลผิดกฎธุรกิจ

    การ merge ก่อน cleaning อาจทำให้เกิดการจับคู่ที่ไม่ถูกต้อง
    จำนวนแถวเพิ่มจาก duplicate key หรือทำให้ข้อมูลที่ invalid
    ถูกนำไปใช้ในการวิเคราะห์

    ดังนั้นควรทำ schema alignment, cleaning,
    standardization, deduplication และ validation
    ก่อน merge เพื่อให้ควบคุม cardinality และตรวจสอบ
    unmatched keys ได้อย่างชัดเจน
"""
)


# ============================================================
# 33) OUTPUT FILE LIST
# ============================================================

print("\n========== OUTPUT FILES ==========")

output_files = [
    "dim_customer.csv",
    "dim_product.csv",
    "fact_sales.csv",
    "data_quality_report.csv",
    "summary_by_province.csv",
    "summary_by_category.csv"
]

for filename in output_files:

    file_path = OUTPUT / filename

    print(
        f"{filename} -> "
        f"{file_path.stat().st_size:,} bytes"
    )


print("\n==========================================")
print("PIPELINE COMPLETE")
print("==========================================")

print(
    "Output folder:",
    OUTPUT
)

TECHTROVE DATA INTEGRATION PIPELINE

Current folder: /content
Output folder : /content/output

========== CHECK FILES ==========
orders_2026_01.csv -> OK
orders_2026_02.csv -> OK
customers_crm.csv -> OK
product_master.xlsx -> OK
payments.json -> OK

========== EXTRACT ==========
January Orders : (361, 8)
February Orders: (391, 8)
Customers      : (163, 5)
Products       : (40, 5)
Payments       : (752, 5)

orders_2026_01.csv
Shape: (361, 8)

Columns:
['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'channel']

Dtype:
order_id        object
order_date      object
customer_id     object
product_id      object
quantity         int64
unit_price     float64
discount       float64
channel         object
dtype: object

Missing:


,missing
order_id,0
order_date,0
customer_id,0
product_id,0
quantity,0
unit_price,1
discount,0
channel,0



Duplicate Rows: 1

Sample:


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,channel
0,ORD000001,2026-01-08 17:11:00,C0158,P039,2,940.5,0.00,Marketplace
1,ORD000002,2026-01-12 20:13:00,C0123,P038,1,12900.0,0.10,Mobile App
2,ORD000003,2026-01-26 14:00:00,C0119,P031,3,1415.5,0.05,Web
3,ORD000004,2026-01-18 16:18:00,C0065,P003,2,25900.0,0.00,Mobile App
4,ORD000005,2026-01-08 17:43:00,C0129,P007,2,23310.0,0.05,Mobile App



orders_2026_02.csv
Shape: (391, 8)

Columns:
['order_id', 'ordered_at', 'customer_id', 'product_id', 'qty', 'unit_price', 'discount_pct', 'channel']

Dtype:
order_id         object
ordered_at       object
customer_id      object
product_id       object
qty               int64
unit_price      float64
discount_pct     object
channel          object
dtype: object

Missing:


...


========== DIM CUSTOMER (10 ROWS) ==========


,customer_id,full_name,email,province,signup_date
0,C0001,ลูกค้า 001,customer001@example.com,ชลบุรี,2024-07-29
1,C0002,ลูกค้า 002,customer002@example.com,ชลบุรี,2025-01-04
2,C0003,ลูกค้า 003,customer003@example.com,ขอนแก่น,2025-10-10
3,C0004,ลูกค้า 004,customer004@example.com,กรุงเทพมหานคร,2024-07-11
4,C0005,ลูกค้า 005,customer005@example.com,ระยอง,2025-06-27
5,C0006,ลูกค้า 006,customer006@example.com,ระยอง,2024-07-13
6,C0007,ลูกค้า 007,customer007@example.com,กรุงเทพมหานคร,2025-09-27
7,C0008,ลูกค้า 008,customer008@example.com,ขอนแก่น,2025-05-27
8,C0009,ลูกค้า 009,customer009@example.com,ภูเก็ต,2024-09-06
9,C0010,ลูกค้า 010,customer010@example.com,ชลบุรี,2024-02-04



========== DIM PRODUCT (10 ROWS) ==========


,product_id,product_name,category,standard_price,active_flag
0,P001,Notebook Model 01,Notebook,299,Y
1,P002,Smartphone Model 02,Smartphone,1490,Y
2,P003,Smartphone Model 03,Smartphone,25900,Y
3,P004,Smart Home Model 04,Smart Home,1490,Y
4,P005,Smartphone Model 05,Smartphone,299,Y
5,P006,Accessory Model 06,Accessory,299,Y
6,P007,Notebook Model 07,Notebook,25900,Y
7,P008,Accessory Model 08,Accessory,499,Y
8,P009,Notebook Model 09,Notebook,299,Y
9,P010,Smart Home Model 10,Smart Home,990,Y




                    6 ANALYSIS ANSWERS

7. หลังรวมไฟล์ orders มี 752 แถว
   และเหลือ 750 แถวหลังลบ duplicate

8. มี customer_id ที่ไม่พบใน Master Data
   จำนวน 22 แถว
   และมี product_id ที่ไม่พบ
   จำนวน 2 แถว

9. มียอดขายที่ใช้ได้จริง
   จำนวน 660 ธุรกรรม
   ยอดขายสุทธิรวม 10,224,044.09 บาท

10. จังหวัดที่มียอดขายสุทธิสูงสุดคือ
    กรุงเทพมหานคร
    ยอดขาย 2,612,955.88 บาท

11. หมวดสินค้าที่มียอดขายสุทธิสูงสุดคือ
    Smartphone
    ยอดขาย 3,092,117.34 บาท

12. หากสลับลำดับ merge ก่อน cleaning
    ความน่าเชื่อถือของข้อมูลอาจลดลง เพราะข้อมูลดิบยังมี
    duplicate key, รูปแบบข้อความที่ไม่เป็นมาตรฐาน,
    schema ที่แตกต่างกันระหว่างเดือน และข้อมูลผิดกฎธุรกิจ

    การ merge ก่อน cleaning อาจทำให้เกิดการจับคู่ที่ไม่ถูกต้อง
    จำนวนแถวเพิ่มจาก duplicate key หรือทำให้ข้อมูลที่ invalid
    ถูกนำไปใช้ในการวิเคราะห์

    ดังนั้นควรทำ schema alignment, cleaning,
    standardization, deduplication และ validation
    ก่อน merge เพื่อให้ควบคุม cardinality และตรวจสอบ
    unmatched keys ได้อย่างชั